# 2-1 세 모델 비교 학습

- 비교 대상: `mobilenet_v3_large`, `efficientnet_b0`, `resnet18`
- `pretrained + backbone freeze` 조건을 동일하게 맞춥니다.
- `Attentive` 클래스는 loss weight를 더 크게 둬서 학습합니다.


In [1]:
from pathlib import Path
import copy
import random
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from sklearn.metrics import accuracy_score, classification_report, f1_score
from tqdm.auto import tqdm


c:\Projects\_active\focus_on_class\worktrees\3_compare\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
data_path = Path("dataset")
save_path = Path("models/three_model_compare_weighted")
save_path.mkdir(parents=True, exist_ok=True)

image_size = 224
batch_size = 16
epochs = 20
learning_rate = 1e-4
weight_decay = 1e-4
seed = 42
freeze_backbone = True
run_final_test = True

candidate_models = [
    "mobilenet_v3_large",
    "efficientnet_b0",
    "resnet18",
]

attentive_boost = 1.4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [3]:
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_data = datasets.ImageFolder(data_path / "train", transform=train_transform)
val_data = datasets.ImageFolder(data_path / "val", transform=test_transform)
test_data = datasets.ImageFolder(data_path / "test", transform=test_transform)

class_names = train_data.classes
class_count = len(class_names)

print(class_names)
print("train:", len(train_data), "val:", len(val_data), "test:", len(test_data))

val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)


['Attentive', 'Drowsy', 'LookingAway']
train: 1345 val: 171 test: 167


In [4]:
class_counts = torch.tensor([
    sum(label == idx for label in train_data.targets)
    for idx in range(class_count)
], dtype=torch.float32)

# 기본은 역빈도 가중치이고, Attentive는 추가 boost를 줍니다.
base_class_weights = class_counts.sum() / (class_count * class_counts)
class_weight_map = {
    name: float(base_class_weights[idx])
    for idx, name in enumerate(class_names)
}

if "Attentive" in class_weight_map:
    class_weight_map["Attentive"] *= attentive_boost

class_weights = torch.tensor(
    [class_weight_map[name] for name in class_names],
    dtype=torch.float32,
    device=device,
)

print("class counts:")
for name, count in zip(class_names, class_counts.tolist()):
    print(f"- {name}: {int(count)}")

print("\nloss weights:")
for name, weight in zip(class_names, class_weights.tolist()):
    print(f"- {name}: {weight:.4f}")


class counts:
- Attentive: 426
- Drowsy: 468
- LookingAway: 451

loss weights:
- Attentive: 1.4734
- Drowsy: 0.9580
- LookingAway: 0.9941


In [5]:
def change_fc(model, class_count):
    model.fc = nn.Linear(model.fc.in_features, class_count)
    return model


def change_classifier(model, class_count):
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, class_count)
    return model


def freeze_model_backbone(model):
    for param in model.parameters():
        param.requires_grad = False
    return model


model_list = {
    "mobilenet_v3_large": [models.mobilenet_v3_large, models.MobileNet_V3_Large_Weights.IMAGENET1K_V2, change_classifier],
    "efficientnet_b0": [models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1, change_classifier],
    "resnet18": [models.resnet18, models.ResNet18_Weights.IMAGENET1K_V1, change_fc],
}


def create_model(model_name, class_count):
    model_func, weights, change_last_layer = model_list[model_name]
    model = model_func(weights=weights)
    if freeze_backbone:
        model = freeze_model_backbone(model)
    model = change_last_layer(model, class_count)
    return model.to(device)


def train_one_epoch(model, optimizer, loss_fn, train_loader, train_data, device):
    model.train()
    total_loss = 0
    y_true = []
    y_pred = []

    for x, y in tqdm(train_loader, leave=False):
        x = x.to(device)
        y = y.to(device)

        pred = model(x)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        y_true += y.cpu().tolist()
        y_pred += pred.argmax(1).cpu().tolist()

    train_loss = total_loss / len(train_data)
    train_acc = accuracy_score(y_true, y_pred)
    train_f1 = f1_score(y_true, y_pred, average="macro")
    return train_loss, train_acc, train_f1


def evaluate(model, loss_fn, loader, data, device):
    model.eval()
    total_loss = 0
    y_true = []
    y_pred = []

    with torch.no_grad():
        for x, y in tqdm(loader, leave=False):
            x = x.to(device)
            y = y.to(device)

            pred = model(x)
            loss = loss_fn(pred, y)

            total_loss += loss.item() * x.size(0)
            y_true += y.cpu().tolist()
            y_pred += pred.argmax(1).cpu().tolist()

    eval_loss = total_loss / len(data)
    eval_acc = accuracy_score(y_true, y_pred)
    eval_f1 = f1_score(y_true, y_pred, average="macro")
    return eval_loss, eval_acc, eval_f1, y_true, y_pred


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def measure_inference_time(model, loader, device, repeat_batches=10):
    model.eval()
    times = []

    with torch.no_grad():
        for i, (x, _) in enumerate(loader):
            if i >= repeat_batches:
                break

            x = x.to(device)
            if device.type == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()
            _ = model(x)
            if device.type == "cuda":
                torch.cuda.synchronize()
            elapsed = time.perf_counter() - start
            times.append(elapsed / x.size(0) * 1000)

    return float(np.mean(times)) if times else np.nan


In [6]:
def run_experiment(model_name):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    train_shuffle_seed = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
        generator=train_shuffle_seed,
    )

    model = create_model(model_name, class_count)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    history = []
    best_score = float("-inf")
    best_epoch = 0
    best_model = copy.deepcopy(model.state_dict())
    train_start = time.perf_counter()

    for epoch in range(1, epochs + 1):
        train_loss, train_acc, train_f1 = train_one_epoch(
            model=model,
            optimizer=optimizer,
            loss_fn=loss_fn,
            train_loader=train_loader,
            train_data=train_data,
            device=device,
        )

        val_loss, val_acc, val_f1, _, _ = evaluate(
            model=model,
            loss_fn=loss_fn,
            loader=val_loader,
            data=val_data,
            device=device,
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1,
        })

        print(
            f"[{model_name}] epoch {epoch:02d}/{epochs} | "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, train_f1={train_f1:.4f} | "
            f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}, val_f1={val_f1:.4f}"
        )

        if val_f1 > best_score:
            best_score = val_f1
            best_epoch = epoch
            best_model = copy.deepcopy(model.state_dict())

    train_time_sec = time.perf_counter() - train_start
    model.load_state_dict(best_model)

    history_df = pd.DataFrame(history)
    best_row = history_df.loc[history_df["val_f1"].idxmax()].to_dict()
    param_count = count_parameters(model)
    val_inference_ms_per_image = measure_inference_time(model, val_loader, device)

    checkpoint_path = save_path / f"{model_name}_best.pt"
    history_path = save_path / f"{model_name}_history.csv"

    torch.save(
        {
            "model_state": model.state_dict(),
            "class_names": class_names,
            "image_size": image_size,
            "model_name": model_name,
            "best_val": best_row,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "batch_size": batch_size,
            "epochs": epochs,
            "seed": seed,
            "freeze_backbone": freeze_backbone,
            "attentive_boost": attentive_boost,
            "class_weight_map": class_weight_map,
            "param_count": param_count,
            "val_inference_ms_per_image": val_inference_ms_per_image,
        },
        checkpoint_path,
    )
    history_df.to_csv(history_path, index=False, encoding="utf-8-sig")

    result = {
        "model_name": model_name,
        "best_epoch": best_epoch,
        "best_val_loss": best_row["val_loss"],
        "best_val_acc": best_row["val_acc"],
        "best_val_f1": best_row["val_f1"],
        "param_count": param_count,
        "train_time_sec": train_time_sec,
        "val_inference_ms_per_image": val_inference_ms_per_image,
        "attentive_weight": class_weight_map.get("Attentive", np.nan),
        "checkpoint_path": str(checkpoint_path),
        "history_path": str(history_path),
    }
    return result, history_df


all_results = []
all_histories = []

for name in candidate_models:
    result, history_df = run_experiment(name)
    all_results.append(result)
    all_histories.append(history_df.assign(model_name=name))

summary_df = pd.DataFrame(all_results)
summary_df = summary_df.sort_values("best_val_f1", ascending=False).reset_index(drop=True)
summary_df


[mobilenet_v3_large] epoch 01/20 | train_loss=1.0585, train_acc=0.4610, train_f1=0.4572 | val_loss=1.0941, val_acc=0.3509, val_f1=0.2311


[mobilenet_v3_large] epoch 02/20 | train_loss=0.9625, train_acc=0.5911, train_f1=0.5782 | val_loss=0.9809, val_acc=0.4678, val_f1=0.4189


[mobilenet_v3_large] epoch 03/20 | train_loss=0.8951, train_acc=0.6454, train_f1=0.6335 | val_loss=0.8886, val_acc=0.5848, val_f1=0.5674


[mobilenet_v3_large] epoch 04/20 | train_loss=0.8403, train_acc=0.6743, train_f1=0.6605 | val_loss=0.8336, val_acc=0.6667, val_f1=0.6566


[mobilenet_v3_large] epoch 05/20 | train_loss=0.8046, train_acc=0.6855, train_f1=0.6734 | val_loss=0.7938, val_acc=0.7251, val_f1=0.7178


...

[resnet18] epoch 17/20 | train_loss=0.5814, train_acc=0.7747, train_f1=0.7673 | val_loss=0.5749, val_acc=0.7251, val_f1=0.6895


[resnet18] epoch 18/20 | train_loss=0.5585, train_acc=0.7985, train_f1=0.7937 | val_loss=0.5398, val_acc=0.7895, val_f1=0.7760


[resnet18] epoch 19/20 | train_loss=0.5589, train_acc=0.7844, train_f1=0.7782 | val_loss=0.5175, val_acc=0.8363, val_f1=0.8286


[resnet18] epoch 20/20 | train_loss=0.5415, train_acc=0.8052, train_f1=0.8006 | val_loss=0.5503, val_acc=0.7427, val_f1=0.7288


,model_name,best_epoch,best_val_loss,best_val_acc,best_val_f1,param_count,train_time_sec,val_inference_ms_per_image,attentive_weight,checkpoint_path,history_path
0,resnet18,19,0.517527,0.836257,0.828565,11178051,131.564672,0.716832,1.473396,models\three_model_compare_weighted\resnet18_b...,models\three_model_compare_weighted\resnet18_h...
1,mobilenet_v3_large,19,0.517872,0.830409,0.826493,4205875,85.836793,0.636564,1.473396,models\three_model_compare_weighted\mobilenet_...,models\three_model_compare_weighted\mobilenet_...
2,efficientnet_b0,19,0.571383,0.807018,0.803669,4011391,85.849859,0.770594,1.473396,models\three_model_compare_weighted\efficientn...,models\three_model_compare_weighted\efficientn...


In [7]:
summary_path = save_path / "three_model_compare_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

all_history_df = pd.concat(all_histories, ignore_index=True)
all_history_path = save_path / "three_model_compare_all_history.csv"
all_history_df.to_csv(all_history_path, index=False, encoding="utf-8-sig")

print(summary_path)
print(all_history_path)
summary_df


models\three_model_compare_weighted\three_model_compare_summary.csv
models\three_model_compare_weighted\three_model_compare_all_history.csv


,model_name,best_epoch,best_val_loss,best_val_acc,best_val_f1,param_count,train_time_sec,val_inference_ms_per_image,attentive_weight,checkpoint_path,history_path
0,resnet18,19,0.517527,0.836257,0.828565,11178051,131.564672,0.716832,1.473396,models\three_model_compare_weighted\resnet18_b...,models\three_model_compare_weighted\resnet18_h...
1,mobilenet_v3_large,19,0.517872,0.830409,0.826493,4205875,85.836793,0.636564,1.473396,models\three_model_compare_weighted\mobilenet_...,models\three_model_compare_weighted\mobilenet_...
2,efficientnet_b0,19,0.571383,0.807018,0.803669,4011391,85.849859,0.770594,1.473396,models\three_model_compare_weighted\efficientn...,models\three_model_compare_weighted\efficientn...


In [10]:
if run_final_test:
    best_checkpoint_path = Path(summary_df.loc[0, "checkpoint_path"])
    checkpoint = torch.load(best_checkpoint_path, map_location=device)

    final_model = create_model(
        model_name=checkpoint["model_name"],
        class_count=len(checkpoint["class_names"]),
    )
    final_model.load_state_dict(checkpoint["model_state"])
    final_model.to(device)

    final_loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    test_loss, test_acc, test_f1, y_true, y_pred = evaluate(
        model=final_model,
        loss_fn=final_loss_fn,
        loader=test_loader,
        data=test_data,
        device=device,
    )

    print(f"best model: {checkpoint['model_name']}")
    print(f"test loss: {test_loss:.4f}")
    print(f"test acc : {test_acc:.4f}")
    print(f"test f1  : {test_f1:.4f}")
    print()
    print(classification_report(y_true, y_pred, target_names=class_names))
else:
    print("run_final_test = False 이므로 test 평가는 생략합니다.")


best model: resnet18
test loss: 0.4690
test acc : 0.8743
test f1  : 0.8700

              precision    recall  f1-score   support

   Attentive       0.86      0.96      0.91        53
      Drowsy       0.85      0.97      0.90        58
 LookingAway       0.93      0.70      0.80        56

    accuracy                           0.87       167
   macro avg       0.88      0.87      0.87       167
weighted avg       0.88      0.87      0.87       167

